# Generating Text using Character RNN

In [1]:
import tensorflow as tf

filepath = tf.keras.utils.get_file(
    fname="shakespeare.txt", 
    origin="https://homl.info/shakespeare",
    cache_dir='datasets',
)

with open(filepath) as f:
    shakespeare_text = f.read()

print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [2]:
# Maps every character into an integer. Starting at 2. Values 0 and 1 are reserved for: padding tokens and unknown characters.
text_vec_layer = tf.keras.layers.TextVectorization(
    split="character", 
    standardize="lower"
)

text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]
encoded -= 2 # drop tokens reserved for 0 (pad) and 1 (unknown), they are not used
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded)

print("encoded ", encoded)
print("n_tokens ", n_tokens)
print("dataset_size ", dataset_size)

encoded  tf.Tensor([19  5  8 ... 20 26 10], shape=(1115394,), dtype=int64)
n_tokens  39
dataset_size  1115394


In [ ]:
from scripts.character_rrn import to_dataset

SEED = 42
MODEL_PATH = "../models/shakespeare_model.keras"

tf.random.set_seed(SEED)

length = 100
train_end = int(dataset_size * 0.90) # 90% for training
val_end = train_end + int(dataset_size * 0.05)

train_set = to_dataset(sequence=encoded[:train_end], length=length, shuffle=True, seed=SEED)
val_set = to_dataset(sequence=encoded[train_end:val_end], length=length)
test_set = to_dataset(sequence=encoded[val_end:], length=length)

In [4]:
train_set

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"]
)

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    MODEL_PATH, 
    monitor="val_accuracy",
    save_best_only=True,
)

history = model.fit(
    train_set, 
    validation_data=val_set, 
    epochs=10, 
    callbacks=[model_ckpt]
)


Epoch 1/10
  31367/Unknown 1911s 60ms/step - accuracy: 0.5471 - loss: 1.4988

c:\Users\cimad\Code\hands-on-nlp\.venv\lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1938s 61ms/step - accuracy: 0.5772 - loss: 1.3790 - val_accuracy: 0.5341 - val_loss: 1.5983
Epoch 2/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1606s 51ms/step - accuracy: 0.5973 - loss: 1.2934 - val_accuracy: 0.5416 - val_loss: 1.5738
Epoch 3/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1524s 48ms/step - accuracy: 0.6009 - loss: 1.2757 - val_accuracy: 0.5450 - val_loss: 1.5654
Epoch 4/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1512s 48ms/step - accuracy: 0.6031 - loss: 1.2664 - val_accuracy: 0.5450 - val_loss: 1.5562
Epoch 5/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1523s 48ms/step - accuracy: 0.6044 - loss: 1.2607 - val_accuracy: 0.5447 - val_loss: 1.5554
Epoch 6/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1519s 48ms/step - accuracy: 0.6055 - loss: 1.2559 - val_accuracy: 0.5440 - val_loss: 1.5559
Epoch 7/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1524s 48ms/step - accuracy: 0.6063 - loss: 1.2523 - val_accuracy: 0.5454 - val_loss: 1.5541
Epoch 8/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1520s 48ms/step

### Model loading

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)

### Model wrapping with preprocessing layers

In [8]:
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda x: x - 2),
    model
])

### One character prediction

In [ ]:
to_be_or_not_to_b = tf.constant(["To be or not to b"])
y_proba = shakespeare_model(to_be_or_not_to_b)[0, -1]
y_pred = tf.argmax(y_proba)
text_vec_layer.get_vocabulary()[y_pred + 2]

np.str_('e')

### Sampling synthetic characters using estimated probabilities

In [12]:
log_probas = tf.math.log([[0.5, 0.4, 0.1]]) 
tf.random.categorical(log_probas, num_samples=8)

<tf.Tensor: shape=(1, 8), dtype=int64, numpy=array([[0, 1, 0, 2, 1, 0, 0, 1]])>

##### Generate some text with different temperatures

In [14]:
from scripts.character_rrn import extend_text

print(extend_text(to_be_or_not_to_b, shakespeare_model, text_vec_layer.get_vocabulary(), temperature=0.01))

ImportError: cannot import name 'extend_text' from 'scripts.character_rrn' (c:\Users\cimad\Code\hands-on-nlp\notebooks\scripts\character_rrn.py)

In [ ]:
print(extend_text(to_be_or_not_to_b, shakespeare_model, text_vec_layer.get_vocabulary(), temperature=1))

In [ ]:
print(extend_text(to_be_or_not_to_b, shakespeare_model, text_vec_layer.get_vocabulary(), temperature=100))